In [1]:
from rdkit.Chem.rdmolfiles import MolToSmiles, MolFromSmiles
from rdkit import Chem
import pandas as pd
import re
from collections import defaultdict
import json

In [2]:
db_app = pd.read_csv('drugbank_approved.csv', sep=',')
db_inv = pd.read_csv('drugbank_investigational.csv', sep=',')

In [3]:
db_app

,DrugBank ID,Name,CAS Number,Drug Groups,InChIKey,InChI,SMILES,Formula,KEGG Compound ID,KEGG Drug ID,PubChem Compound ID,PubChem Substance ID,ChEBI ID,ChEMBL ID,HET ID,ChemSpider ID,BindingDB ID
0,DB00006,Bivalirudin,128270-60-0,approved; investigational,OIRCOABEOLEUMC-GEJPAHFPSA-N,InChI=1S/C98H138N24O33/c1-5-52(4)82(96(153)122...,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,C98H138N24O33,NaN,D03136,16129704.0,46507415.0,59173.0,CHEMBL2103749,NaN,10482069.0,50248103.0
1,DB00014,Goserelin,65807-02-5,approved,BLCLNMBMMGCOAS-URPVMXJPSA-N,InChI=1S/C59H84N18O14/c1-31(2)22-40(49(82)68-3...,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,C59H84N18O14,NaN,D00573,5311128.0,46507336.0,5523.0,CHEMBL1201247,NaN,4470656.0,NaN
2,DB00027,Gramicidin D,1405-97-6,approved,NDAYQJDHGXTBJL-MWWSRJDJSA-N,InChI=1S/C96H135N19O16/c1-50(2)36-71(105-79(11...,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,C96H135N19O16,NaN,D04369,45267103.0,46507412.0,NaN,CHEMBL557217,NaN,24623445.0,NaN
3,DB00035,Desmopressin,16679-58-6,approved,NFLWUMRGJYTJIN-PNIOQBSNSA-N,InChI=1S/C46H64N14O12S2/c47-35(62)15-14-29-40(...,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,C46H64N14O12S2,C06944,D00291,NaN,NaN,4450.0,CHEMBL1429,NaN,4470602.0,50205308.0
4,DB00050,Cetrorelix,120287-85-6,approved; investigational,SBNPWPIBESPSIF-MHWMIDJBSA-N,InChI=1S/C70H92ClN17O14/c1-39(2)31-52(61(94)82...,CC(C)C[C@H](NC(=O)[C@@H](CCCNC(N)=O)NC(=O)[C@H...,C70H92ClN17O14,NaN,D07665,25074887.0,46505494.0,59224.0,CHEMBL1200490,NaN,10482082.0,50369965.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2797,DB19375,Chlorothymol,89-68-9,approved,KFZXVMNBUMVKLN-UHFFFAOYSA-N,InChI=1S/C10H13ClO/c1-6(2)8-5-9(11)7(3)4-10(8)...,CC(C)C1=CC(Cl)=C(C)C=C1O,C10H13ClO,NaN,NaN,NaN,NaN,NaN,CHEMBL1441646,NaN,NaN,NaN
2798,DB19376,Salicylanilide,87-17-2,approved; withdrawn,WKEDVNSFRWHDNR-UHFFFAOYSA-N,InChI=1S/C13H11NO2/c15-12-9-5-4-8-11(12)13(16)...,OC1=CC=CC=C1C(=O)NC1=CC=CC=C1,C13H11NO2,C18915,NaN,NaN,NaN,239133.0,CHEMBL82970,SLI,NaN,234300.0
2799,DB19378,Megestrol,3562-63-8,approved,VXIMPSPISRVBPZ-NWUMPJBXSA-N,InChI=1S/C22H30O3/c1-13-11-16-17(20(3)8-5-15(2...,[H][C@@]12CC[C@](O)(C(C)=O)[C@@]1(C)CC[C@@]1([...,C22H30O3,C07120,NaN,NaN,NaN,6722.0,CHEMBL4071215,NaN,NaN,NaN
2800,DB19379,Syrosingopine,84-36-6,approved; withdrawn,ZCDNRPPFBQDQHR-SSYATKPKSA-N,InChI=1S/C35H42N2O11/c1-7-46-35(40)48-31-26(42...,[H][C@]12C[C@@H](OC(=O)C3=CC(OC)=C(OC(=O)OCC)C...,C35H42N2O11,NaN,NaN,NaN,NaN,32175.0,CHEMBL1399124,NaN,NaN,NaN


In [4]:
db_inv

,DrugBank ID,Name,CAS Number,Drug Groups,InChIKey,InChI,SMILES,Formula,KEGG Compound ID,KEGG Drug ID,PubChem Compound ID,PubChem Substance ID,ChEBI ID,ChEMBL ID,HET ID,ChemSpider ID,BindingDB ID
0,DB00006,Bivalirudin,128270-60-0,approved; investigational,OIRCOABEOLEUMC-GEJPAHFPSA-N,InChI=1S/C98H138N24O33/c1-5-52(4)82(96(153)122...,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,C98H138N24O33,NaN,D03136,16129704.0,46507415.0,59173.0,CHEMBL2103749,NaN,10482069.0,50248103.0
1,DB00050,Cetrorelix,120287-85-6,approved; investigational,SBNPWPIBESPSIF-MHWMIDJBSA-N,InChI=1S/C70H92ClN17O14/c1-39(2)31-52(61(94)82...,CC(C)C[C@H](NC(=O)[C@@H](CCCNC(N)=O)NC(=O)[C@H...,C70H92ClN17O14,NaN,D07665,25074887.0,46505494.0,59224.0,CHEMBL1200490,NaN,10482082.0,50369965.0
2,DB00080,Daptomycin,103060-53-3,approved; investigational,DOAKLVKFURWEDJ-QCMAZARJSA-N,InChI=1S/C72H101N17O26/c1-5-6-7-8-9-10-11-22-5...,CCCCCCCCCC(=O)N[C@@H](CC1=CNC2=C1C=CC=C2)C(=O)...,C72H101N17O26,C12013,D01080,16134395.0,46504551.0,600103.0,CHEMBL4744444,NaN,10200644.0,NaN
3,DB00091,Cyclosporine,59865-13-3,approved; investigational; vet_approved,PMATZTZNYRCHOR-CGLBZJNRSA-N,InChI=1S/C62H111N11O12/c1-25-27-28-40(15)52(75...,CC[C@@H]1NC(=O)[C@H]([C@H](O)[C@H](C)C\C=C\C)N...,C62H111N11O12,C05086,D00184,5284373.0,46508198.0,4031.0,CHEMBL160,NaN,4447449.0,50022815.0
4,DB00106,Abarelix,183552-38-7,approved; investigational; withdrawn,AIWRTTMUVOZGPW-HSPKUQOVSA-N,InChI=1S/C72H95ClN14O14/c1-41(2)32-54(64(93)80...,CC(C)C[C@H](NC(=O)[C@@H](CC(N)=O)NC(=O)[C@H](C...,C72H95ClN14O14,NaN,D02738,16131215.0,46508237.0,337298.0,CHEMBL1252,NaN,10482301.0,50102442.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5296,DB19450,TDI-01,NaN,investigational,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5297,DB19451,Ibuzatrelvir,2755812-39-4,investigational,WGNWEPPRWQKSKI-AIEDFZFUSA-N,"InChI=1S/C21H30F3N5O5/c1-20(2,3)15(28-19(33)34...",COC(=O)N[C@H](C(=O)N1C[C@@H](C[C@H]1C(=O)N[C@@...,C21H30F3N5O5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5298,DB19454,Cetyl oleate,22393-86-8,investigational,JYTMDBGMUIAIQH-ZPHPHTNESA-N,InChI=1S/C34H66O2/c1-3-5-7-9-11-13-15-17-19-20...,CCCCCCCCCCCCCCCCOC(=O)CCCCCCC\C=C/CCCCCCCC,C34H66O2,NaN,NaN,NaN,NaN,75622.0,NaN,NaN,NaN,NaN
5299,DB19455,Cetyl myristoleate,64660-84-0,investigational,DYIOQMKBBPSAFY-BENRWUELSA-N,InChI=1S/C30H58O2/c1-3-5-7-9-11-13-15-16-17-19...,CCCCCCCCCCCCCCCCOC(=O)CCCCCCC\C=C/CCCC,C30H58O2,NaN,NaN,NaN,NaN,165719.0,NaN,NaN,NaN,NaN


In [5]:
overlapped = []
db_inv_id = db_inv['DrugBank ID'].to_list()
db_app_id = db_app['DrugBank ID'].to_list()
for id in db_inv_id:
    if id in db_app_id:
        db_inv_id.remove(id)
print('Number of unique investigational drugs:', len(db_inv_id))

Number of unique investigational drugs: 4535


In [6]:
unique_inv = db_inv[db_inv['DrugBank ID'].isin(db_inv_id)]

In [7]:
df_db = pd.concat([db_app,unique_inv],axis=0)

In [8]:
db_smiles = list(set(df_db['SMILES'].to_list()))
print('Number of smiles:', len(db_smiles))

Number of smiles: 6116


In [9]:
processed_smiles = []
warning_smiles = []
for smiles in db_smiles:
    try:
        mol = MolFromSmiles(smiles)
        Chem.Kekulize(mol)
        new_smile = MolToSmiles(mol, isomericSmiles=False)
        processed_smiles.append(new_smile)
    except:
        warning_smiles.append(new_smile)
processed_smiles.extend(warning_smiles)

[14:31:11] Conflicting single bond directions around double bond at index 46.
[14:31:11]   BondStereo set to STEREONONE and single bond directions set to NONE.
[14:31:11] WARNING: not removing hydrogen atom without neighbors
[14:31:11] WARNING: not removing hydrogen atom without neighbors
[14:31:11] Explicit valence for atom # 1 Cl, 4, is greater than permitted
[14:31:11] Explicit valence for atom # 1 B, 6, is greater than permitted
[14:31:11] WARNING: not removing hydrogen atom without neighbors
[14:31:11] WARNING: not removing hydrogen atom without neighbors
[14:31:11] WARNING: not removing hydrogen atom without neighbors
[14:31:11] WARNING: not removing hydrogen atom without neighbors
[14:31:11] WARNING: not removing hydrogen atom without neighbors
[14:31:11] SMILES Parse Error: syntax error while parsing: OC1=CC=CC(=C1)C-1=C2\CCC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC(O)=CC=C1)C1=CC(O)=CC=C1)\C1=CC(O)=CC=C1
[14:31:11] SMILES Parse Error: Failed parsing SMILES 

In [10]:
def has_ring(smile:str):
    return any(s.isdigit() for s in smile)

In [11]:
for i in range(len(processed_smiles)):
    processed_smiles[i] = re.sub(r'Cl', 'L', processed_smiles[i])

    processed_smiles[i] = processed_smiles[i].upper()

    if any(s.isdigit() for s in processed_smiles[i]):
        processed_smiles[i] = re.sub(r'\d', 'X', processed_smiles[i], count=1)

    processed_smiles[i] = re.sub(r'C\(F\)\(F\)\(F\)', 'W', processed_smiles[i])

    processed_smiles[i] = re.sub(r'S\(=O\)\(=O\)', 'U', processed_smiles[i])

    processed_smiles[i] = re.sub(r'S\(=O\)(?!\(=O\))', 'M', processed_smiles[i])

In [12]:
processed_smiles[:5]

['CCCCCCCCCCCCNCC(O)CCCCCCCCCNXC(=O)C2C(NCN2C)N(C)C1=O',
 'CC(O)C(=O)[O-].[K+]',
 'CXCC(N2CCNCC2)C2OCCOC2C1',
 'CC(C)(C)CXCNC(CSC2CNC(NC(=O)C3CCNCC3)S2)O1',
 'CCX2CCC3C4C(CC(OUO)CC4)CCC3C1CCC2=O']

In [13]:
df_smiles = pd.DataFrame(processed_smiles, columns=['SMILES'])
df_smiles.to_csv('db_app_inv_smiles.csv',index=False)

In [14]:
common_ngrams = set()
for smile in processed_smiles:
    ngram_s = set(
        smile[i:i+l]
        for l in range(0,6)
        for i in range(len(smile)-l)
    )
    common_ngrams|=ngram_s
print(len(common_ngrams))

20983


In [15]:
result = defaultdict(lambda:defaultdict(int))
for smile in processed_smiles:
    for ngram in common_ngrams:
        start = 0
        while True:
            idx = smile.find(ngram, start)
            if idx == -1:
                break
            
            if idx + len(ngram) < len(smile):
                next_char = smile[idx + len(ngram)]
                result[ngram][next_char] += 1
            start = idx +1

result_dict = {ngram: dict(next_char) for ngram, next_char in result.items()}

In [16]:
tot_count = {ngram: 0 for ngram in result_dict.keys()}
for ngram, freq_dict in result_dict.items():
    tot_count[ngram] = 0
    for next_char, count in freq_dict.items():
        tot_count[ngram] += count

In [17]:
for ngram, freq_dict in result_dict.items():
    for next_char, count in freq_dict.items():
        freq = count/tot_count[ngram]
        freq_dict[next_char] = freq
    result_dict[ngram] = freq_dict

In [18]:
with open('ngram_db_app_inv.json','w') as f:
    json.dump(result_dict, f)